# Day 4 — Multi-Agent Systems and Evaluation

## Daily project: Engineering Design Review Team

This is the classroom master notebook for Day 4. Work from top to bottom: each section introduces one limitation, adds one system layer, and carries that improvement into the daily project.

### How to use this notebook

- Run the **Environment setup** cell directly below first. On Google Colab it clones the repository, installs packages, and asks for your API key. On your own computer it only loads the `.env` file.
- Every section starts with a small setup cell of its own; if the kernel restarts, rerun that cell and continue.
- Run the code cells in order and read the printed output: each cell prints what changed and why.
- Every lesson ends with a short **Checkpoint** (answers are folded under *Show answer*) and a **Recap**.
- Without an API key everything runs in deterministic **mock** mode and spends no credit. Use the instructor-issued OpenRouter key only for the marked live observations.
- Section 4.8 is the day's single hands-on exercise; a commented reference solution follows its check.

### Day 4 contents

1. [A Review Task We Can Measure](#day-4-section-1)
2. [Single-Reviewer Baseline](#day-4-section-2)
3. [Deterministic Checks Before Model Judgment](#day-4-section-3)
4. [Parallel Specialist Reviewers](#day-4-section-4)
5. [Supervisor Synthesis](#day-4-section-5)
6. [Comparative Evaluation](#day-4-section-6)
7. [Day 4 Project — Engineering Design Review Team](#day-4-section-7)
8. [Pivotal Exercise: Merge Specialist Findings](#day-4-section-8)

---


In [ ]:
# --- Environment setup: run this cell first (Colab or local) -------------------------
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/cto-school/agentic-ai-engineering.git"   # the public course repository
REPO_DIR = Path("/content/agentic-ai-engineering")

if IN_COLAB:
    if not REPO_DIR.exists():
        print("Cloning the course repository ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
        print("Installing requirements (this takes a minute) ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "setup/requirements-core.txt")], check=True)
    os.chdir(REPO_DIR / "day_04_multi_agent_systems")
    # Colab has no .env file. Paste the key you were issued; it is kept only in this runtime.
    from getpass import getpass
    if not os.getenv("OPENROUTER_API_KEY"):
        key = getpass("OPENROUTER_API_KEY (press Enter to stay in mock mode): ").strip()
        if key:
            os.environ["OPENROUTER_API_KEY"] = key
else:
    # Local machine: the key is read from the .env file at the repository root
    # (Day 1.1 explains how to create it from .env.example).
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))

print("Working directory:", os.getcwd())
print("Mode:", "LIVE (OpenRouter key found)" if os.getenv("OPENROUTER_API_KEY") else "MOCK (no key found: deterministic answers, no credit spent)")


<a id="day-4-section-1"></a>

## 4.1 — A Review Task We Can Measure


## Before you begin

### Learning outcomes

- Read a supplied artifact and describe a defect as a structured, evidenced record.
- Explain why a hidden answer key is what turns "the review looked good" into a measurement.
- Predict two defects yourself before the answer key is revealed.

Architecture reference: [Day 4 diagrams D12](../diagrams/source/day_04.md)

### Expected observation

The artifact prints with line numbers, your prediction comes first, and only then does the answer key reveal that nine defects were seeded.

> **Need an API key?** You do not need one today: every cell runs offline. If you want the optional live experiment, create the `.env` file exactly as shown in **Day 1.1 — Your First Model Call**, at the repository root, then restart the kernel.


## Concept briefing

## Why multiple agents are not the starting point

Adding agents adds model calls, duplicated context, coordination logic, latency, cost and
new failure modes. It is justified only when a task splits into bounded perspectives whose
combined quality beats a simpler system by enough to pay for that complexity.

So today we do not argue about it. We build one general reviewer, one reviewer plus
deterministic tools, and a three-specialist team with a supervisor, run all three over the
same artifact with a hidden answer key, and read the numbers.

We run that comparison twice, against two different reviewers: one with real blind spots,
and one that is already strong. The winner is not the same both times. **The single
reviewer is allowed to win, and in one of the two runs it does.**


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact

Everything today reviews the same file. It is deliberately defective classroom code: never copy it into anything real.


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — Read it the way a reviewer would

Line numbers matter. A finding without a location cannot be checked, merged, or argued with, so every finding we produce today will carry one.


In [ ]:
# Print the artifact with line numbers so we can point at defects precisely.
for number, line in enumerate(SOURCE.splitlines(), 1):
    print(f"{number:>2}: {line}")


## Step 3 — What a finding has to contain

`Finding` is the contract every reviewer and the supervisor agree on. Five of its fields are the claim; two are provenance. If a reviewer cannot fill all of them, it does not have a finding — it has an opinion.


In [ ]:
from review_team import Finding

# Build one finding by hand so the contract is concrete before any reviewer runs.
example = Finding(
    id="EXAMPLE-1",                              # identity of this record
    category="security",                         # correctness | security | maintainability
    line=3,                                      # where in the artifact
    title="Credential-like token is hardcoded",  # the claim, in one line
    evidence=SOURCE.splitlines()[2].strip(),     # the exact source it rests on
    severity="high",                             # low | medium | high | critical
    recommendation="Load secrets from an injected secret store.",
    reviewer="you",                              # who said it
)

for field_name, value in example.as_dict().items():
    print(f"{field_name:>15}: {value}")


### Try it yourself

Scroll back to the printed artifact and pick **two** more defects — one you would call `correctness` and one you would call `maintainability`. Write down the line number and the exact source text for each. Then run the worked solution.


In [ ]:
# --- Worked solution ---
# Two defects you can find by reading alone, written as proper Finding records.
lines = SOURCE.splitlines()

my_findings = [
    Finding(
        id="MINE-1",
        category="correctness",
        line=25,                                  # "return sum(...) / len(items)"
        title="Empty item list causes division by zero",
        evidence=lines[24].strip(),               # index 24 == line 25
        severity="medium",
        recommendation="Decide what an empty order should return before dividing.",
        reviewer="student",
    ),
    Finding(
        id="MINE-2",
        category="maintainability",
        line=26,                                  # "except Exception:"
        title="Broad exception handler hides unrelated failures",
        evidence=lines[25].strip(),
        severity="medium",
        recommendation="Catch only the errors you expect, and log the rest.",
        reviewer="student",
    ),
]

for finding in my_findings:
    print(f"line {finding.line:>3} | {finding.category:<15} | {finding.severity:<6} | "
          f"{finding.title}")
    print(f"         evidence: {finding.evidence}")

print("\nFindings written before seeing the answer key:", len(my_findings))


## Step 4 — Now reveal the answer key

The golden set lists every defect that was deliberately seeded. It exists so we can compute **recall** (how many known defects a system found) and **false positives** (claims that match nothing). It is instructor-owned and never goes into a prompt.


In [ ]:
import json

golden = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))
print("Seeded defects:", len(golden), "\n")

for category in ("correctness", "security", "maintainability"):
    in_category = [item for item in golden if item["category"] == category]
    print(f"{category} ({len(in_category)}):")
    for item in in_category:
        print(f"   line {item['line']:>3}  {item['id']}  [{item['severity']}]  {item['title']}")
    print()


## Step 5 — Score your own review

Same scoring code we will use for every agent today. Notice the rule: a defect can be credited **once**. Reporting it twice is a duplicate, not two discoveries.


In [ ]:
from review_team import ReviewRun, evaluate

# Wrap your two findings in a ReviewRun so the evaluator can score them like any system.
my_run = ReviewRun(system="human_reader", findings=my_findings)
row = evaluate(my_run, GOLDEN_PATH)

print("Known defects  :", row["known_defects"])
print("You found      :", row["found"])
print("Recall         :", row["recall"])
print("False positives:", row["false_positives"])
print("You missed     :", row["missed"])


### Checkpoint

**1. Why must the golden set stay out of the reviewer's prompt?**

<details><summary>Show answer</summary>

Because a reviewer that has been shown the answers is being tested on copying, not on reviewing. Its recall would measure prompt leakage instead of capability, and the comparison between architectures would become meaningless.

</details>

**2. Two reviewers both report the `eval` call on line 20. How many defects were found?**

<details><summary>Show answer</summary>

One. Each known defect is credited once; the second report is counted as a *duplicate*. Counting it twice would let a system inflate its recall simply by repeating itself, which is exactly the failure mode a multi-agent team risks.

</details>

### Recap

- Limitation we saw: "the review looked thorough" is not a measurement — nothing in it can be compared between two systems.
- Layer we added: a structured `Finding` contract plus an instructor-owned golden defect set, scored by `evaluate`.
- Evidence it worked: your two hand-written findings were scored automatically, with recall, false positives and a list of what you missed.


---

### Section 4.1 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-2"></a>

## 4.2 — Single-Reviewer Baseline


## Before you begin

### Learning outcomes

- Call a reviewer through one provider contract that both a real model and an offline mock satisfy.
- Read the telemetry a run actually produced instead of an estimate.
- Measure the baseline that every later architecture must beat.

Architecture reference: [Day 4 diagrams D12](../diagrams/source/day_04.md)

### Expected observation

One reviewer finds 5 of the 9 seeded defects in mock mode, and the run reports the tokens the provider says it used.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — One contract, two implementations

Every reviewer today — real or mock — is called the same way:

```python
findings, usage = provider.review(source, role)
```

`role` is `"general"` for this baseline. Because the contract is identical, we can swap the reviewer without touching the architecture we are testing.


In [ ]:
# Build the reviewer. This is the ONLY place the notebook decides live vs mock.
from review_team import FallbackReviewer, MockStructuredReviewer, OpenRouterReviewer

SCENARIO = "blind_spots"          # which blind spots the scripted reviewer has

if LIVE:
    # FallbackReviewer tries the real model and, if the call fails for any reason,
    # prints one line and uses the mock for that call so the lesson never stops.
    provider = FallbackReviewer(OpenRouterReviewer(), MockStructuredReviewer(SCENARIO))
else:
    provider = MockStructuredReviewer(SCENARIO)

print("Reviewer provider:", type(provider).__name__)
print("Scenario         :", SCENARIO)


## Step 3 — Run the single reviewer

`run_single_reviewer` makes exactly one call, applies no tools and runs no supervisor. It is the simplest system that could possibly work.


In [ ]:
from review_team import run_single_reviewer

single = run_single_reviewer(SOURCE, provider)

print("System     :", single.system)
print("Model calls:", single.model_calls)
print("Findings   :", len(single.findings), "\n")

for finding in single.findings:
    print(f"line {finding.line:>3} | {finding.category:<15} | {finding.severity:<8} | "
          f"{finding.title}")


## Step 4 — Read the telemetry, do not guess it

The token numbers below come from the provider's own usage report, which the run stored in its trace. A step that never calls a model reports zero — you will see one of those in the next notebook.


In [ ]:
step = single.trace[0]                  # the run's only step
usage = step["usage"]                  # what the provider said it consumed

print("Step             :", step["step"])
print("Model            :", usage["model"])
print("Live call?       :", usage["live"])
print("Prompt tokens    :", usage["prompt_tokens"])
print("Completion tokens:", usage["completion_tokens"])
print("Total tokens     :", single.total_tokens)
print("Cost reported    : $%.6f" % single.cost_usd)
print("Elapsed ms       : %.2f" % single.elapsed_ms)


## Step 5 — Score the baseline

Now the golden set earns its keep. `missed` is the list this whole day exists to shrink — or to prove we cannot shrink it economically.


In [ ]:
from review_team import evaluate

row = evaluate(single, GOLDEN_PATH)

print("Found      : %d / %d" % (row["found"], row["known_defects"]))
print("Recall     :", row["recall"])
print("False positives:", row["false_positives"])
print("Duplicates :", row["duplicates"])
print("Missed     :", row["missed"])


## Step 6 — Be honest about what this reviewer is

In mock mode the reviewer is **scripted**, not intelligent. Its blind spots are a parameter we chose, so the classroom result is reproducible on any laptop with no API key. Nothing here is a claim about how good real language models are at code review — it is a controlled way to compare *architectures* while holding the reviewer constant.


In [ ]:
from review_team import SCENARIOS

for scenario, roles in SCENARIOS.items():
    print(scenario, "-> general reviewer can see", len(roles["general"]), "of 9 defects")
print("\nCurrently using:", SCENARIO)


### Try it yourself

Predict: if we swap in the `strong_generalist` reviewer — same architecture, one call, no specialists — how many of the 9 defects will it find? Write your number down, then run the worked solution.


In [ ]:
# --- Worked solution ---
# Same architecture (run_single_reviewer), different reviewer. Only the blind spots move.
strong_provider = MockStructuredReviewer("strong_generalist")
strong = run_single_reviewer(SOURCE, strong_provider)
strong_row = evaluate(strong, GOLDEN_PATH)

print("blind_spots       reviewer found: %d / 9  (missed %s)"
      % (row["found"], row["missed"]))
print("strong_generalist reviewer found: %d / 9  (missed %s)"
      % (strong_row["found"], strong_row["missed"]))
print()
print("Both runs used", strong.model_calls, "model call. The architecture did not change;")
print("the reviewer did. Remember this when we start adding agents.")


### Checkpoint

**1. Why does every reviewer — mock, live, and the fake one in the tests — go through the same `provider.review(source, role)` contract?**

<details><summary>Show answer</summary>

Because it lets us change one variable at a time. If the mock and the live model plug into the same slot, a difference in results comes from the architecture or the reviewer, never from rewriting the pipeline. It is also what makes an offline, zero-cost classroom path possible.

</details>

**2. The run reports `prompt_tokens` from the provider rather than estimating them from the length of the source. Why does that distinction matter?**

<details><summary>Show answer</summary>

An estimate printed in a results table looks exactly like a measurement. If a step that made no call still reports a plausible token count, every cost comparison built on that table is fiction. Measured telemetry — or an explicit zero labelled "no model call" — is the only honest option.

</details>

### Recap

- Limitation we saw: one general reviewer covering everything found 5 of 9 seeded defects and missed a whole category's worth.
- Layer we added: a single provider contract with real usage telemetry, plus a scored baseline run.
- Evidence it worked: 5/9 recall, 1 model call, and a printed `missed` list that later systems have to shrink.


---

### Section 4.2 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-3"></a>

## 4.3 — Deterministic Checks Before Model Judgment


## Before you begin

### Learning outcomes

- Prove some defects with a parser instead of paying a model to guess at them.
- Combine free deterministic evidence with one bounded model call.
- See a trace step that honestly reports zero tokens.

Architecture reference: [Day 4 diagrams D15](../diagrams/source/day_04.md)

### Expected observation

The AST checker reports 3 findings in well under a millisecond for 0 tokens, and recall rises from 5/9 to 6/9 without a second model call.


## Concept briefing

## Deterministic tools before more model calls

Some findings need no model judgement at all. A Python parser can prove that a file calls
`eval`, that a function has a mutable default argument, and that an `except Exception:`
handler exists. Linters, tests, type checkers and security scanners give objective
evidence for the patterns they support, for free, in milliseconds, with the same answer
every time.

Model reviewers earn their cost on ambiguous intent, cross-cutting reasoning,
prioritisation and explanation. A strong system pairs deterministic evidence with bounded
judgement instead of paying three models to rediscover facts a parser already proved.

One consequence matters for measurement: a static checker has never heard of your answer
key, so it invents its own finding ids. Your evaluator therefore has to match a finding to
a defect by *location*, exactly as it must for a real model.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — Some defects are facts, not opinions

Python can parse itself. `ast` turns the source into a tree, and walking that tree *proves* that the file calls `eval`, that a function has a mutable default argument, and that a bare `except Exception:` exists. No judgement is involved, so no model is needed.


In [ ]:
from review_team import deterministic_checks

checks = deterministic_checks(SOURCE)

print("Deterministic findings:", len(checks), "\n")
for finding in checks:
    print(f"line {finding.line:>3} | {finding.category:<15} | {finding.severity:<8} | "
          f"{finding.title}")
    print(f"         id       : {finding.id}")
    print(f"         evidence : {finding.evidence}")


## Step 3 — Notice the identifiers

The checker's ids start with `AST-`, not `DEF-`. That is deliberate: a real static analyser has never heard of our answer key. So the evaluator has to match a finding to a defect by **location** — same category, close enough line — exactly as it must for a real model whose ids are arbitrary strings.


In [ ]:
from review_team import match_to_golden
import json

golden = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))

for finding in checks:
    matched = match_to_golden(finding, golden)
    print(f"{finding.id:<26} -> golden defect {matched}")


## Step 4 — Measure what the check cost

This is the whole argument for running tools first: the same evidence, for nothing.


In [ ]:
from time import perf_counter

start = perf_counter()
for _ in range(100):
    deterministic_checks(SOURCE)          # run it 100 times so the timer has something to see
elapsed_ms = (perf_counter() - start) * 1000 / 100

print("Average time per AST pass: %.3f ms" % elapsed_ms)
print("Model calls used         : 0")
print("Tokens used              : 0")
print("Cost                     : $0.000000")
print("Result changes between runs: no (same tree, same answer, every time)")


## Step 5 — Checks plus one reviewer

`run_checks_plus_reviewer` runs the parser, then makes **one** model call, then merges both sets of findings. Watch the trace: the first step declares that it made no model call.


In [ ]:
# Build the reviewer. This is the ONLY place the notebook decides live vs mock.
from review_team import FallbackReviewer, MockStructuredReviewer, OpenRouterReviewer

SCENARIO = "blind_spots"          # which blind spots the scripted reviewer has

if LIVE:
    # FallbackReviewer tries the real model and, if the call fails for any reason,
    # prints one line and uses the mock for that call so the lesson never stops.
    provider = FallbackReviewer(OpenRouterReviewer(), MockStructuredReviewer(SCENARIO))
else:
    provider = MockStructuredReviewer(SCENARIO)

print("Reviewer provider:", type(provider).__name__)
print("Scenario         :", SCENARIO)


In [ ]:
from review_team import evaluate, run_checks_plus_reviewer, run_single_reviewer

single    = run_single_reviewer(SOURCE, provider)
augmented = run_checks_plus_reviewer(SOURCE, provider)

print("Trace of the augmented run:")
for step in augmented.trace:
    print(" ", step["step"])
    for key, value in step.items():
        if key != "step":
            print("     ", key, "=", value)


## Step 6 — Did it help, and what did it cost?

Compare the two rows. The AST pass added a defect the reviewer missed, and the model bill did not move at all.


In [ ]:
single_row    = evaluate(single, GOLDEN_PATH)
augmented_row = evaluate(augmented, GOLDEN_PATH)

for label, r in (("single_reviewer", single_row), ("checks_plus_reviewer", augmented_row)):
    print(f"{label:<22} found {r['found']}/9 | calls {r['model_calls']} | "
          f"tokens {r['tokens']} | merged duplicates {r['merged_duplicates']}")

print("\nDefect gained by adding the parser:",
      sorted(set(single_row["missed"]) - set(augmented_row["missed"])))
print("Extra model calls to gain it     :",
      augmented_row["model_calls"] - single_row["model_calls"])
print("Extra tokens to gain it          :",
      augmented_row["tokens"] - single_row["tokens"])


## Step 7 — Where the parser stops

The AST checker cannot tell you that a flat 20-unit discount can push a total negative, or that a quantity should never be negative. Those are business rules, not syntax. That is the boundary where model judgement starts to be worth paying for.


In [ ]:
print("Still missed after the parser ran:")
for defect_id in augmented_row["missed"]:
    item = next(x for x in golden if x["id"] == defect_id)
    print(f"   {defect_id}  line {item['line']:>3}  {item['title']}")
print("\nNone of these are syntax facts; each needs judgement about intent.")


### Try it yourself

The supervisor merged some findings in step 5. Predict: how many of the parser's 3 findings were things the reviewer had *already* reported, and how many were new?


In [ ]:
# --- Worked solution ---
# Merge the two groups by hand and ask the supervisor to report what it did.
from review_team import synthesize_with_report

reviewer_findings, _usage = provider.review(SOURCE, "general")
report = synthesize_with_report([checks, reviewer_findings])

print("Findings that arrived at the supervisor:", report.received)
print("Merged as duplicates                   :", report.merged_duplicates)
print("Kept in the final report               :", report.kept)
print()
new_count = len(checks) - report.merged_duplicates
print("So of the parser's %d findings, %d overlapped what the reviewer had already said"
      % (len(checks), report.merged_duplicates))
print("and %d was new (DEF-MNT-02, the broad exception handler)." % new_count)
print("Overlap is not waste here: the parser's version is *proof*, the reviewer's was a claim.")


### Checkpoint

**1. The parser and the reviewer both reported the `eval` call on line 20. Is that wasted work?**

<details><summary>Show answer</summary>

Not in this case — it is free (0 tokens) and it upgrades a model's claim into a parser's proof. It becomes waste when you pay a *second model call* to rediscover something ordinary code already established. That is the rule: prove what you can prove, then spend model calls on what is left.

</details>

**2. Why does the deterministic step write `"model_calls": 0` into the trace instead of simply leaving the field out?**

<details><summary>Show answer</summary>

Because an absent number gets silently filled in by whoever reads the table next. An explicit zero labelled `no model call` makes the free step visible in every cost comparison, and stops anyone attributing the parser's findings to the model's bill.

</details>

### Recap

- Limitation we saw: the single reviewer missed defects a parser can prove in a fraction of a millisecond.
- Layer we added: an AST checker whose findings carry their own ids, merged with the reviewer's by a bounded supervisor.
- Evidence it worked: recall 5/9 -> 6/9 with 0 extra model calls and 0 extra tokens; the trace shows the free step reporting zeros.


---

### Section 4.3 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-4"></a>

## 4.4 — Parallel Specialist Reviewers


## Before you begin

### Learning outcomes

- Name the four pieces of a fan-out precisely: shared state, fan-out, handoff, fan-in.
- Run three narrow reviewers sequentially, then in threads, and time both.
- Show that parallelism buys wall-clock time and nothing else.

Architecture reference: [Day 4 diagrams D13](../diagrams/source/day_04.md)

### Expected observation

Three specialists each report only their own category; the sequential run takes about 0.9 s and the threaded run about 0.3 s, with identical calls and tokens.


## Concept briefing

## Specialist decomposition

A specialist role must narrow the task, not merely rename the same prompt. Our
correctness, security and maintainability reviewers all read the same immutable artifact
but answer different questions, and all return the same `Finding` contract: category,
location, evidence, severity, recommendation, and which role produced it.

That contract is the **handoff**. It is what crosses the boundary between agents - not
personas, not hidden reasoning, not a full chat transcript. The supervisor does not need
each reviewer's conversation; it needs validated records with enough provenance to resolve
duplicates and conflicts.

Narrowing a prompt does not create knowledge the reviewer never had. If the generalist
cannot see a subtle business rule, the specialist with a narrower prompt usually cannot
see it either. Decomposition redistributes attention; it does not add capability.

## Sequential before parallel

Say the words precisely:

- **Fan-out**: one step launches several independent branches.
- **Shared state**: what all the branches read. Here it is the immutable artifact text.
  Nothing mutates it, which is exactly why the branches are safe to run at the same time.
- **Fan-in**: one step collects every branch's results and combines them.

Run the branches sequentially first, because the order and any failure are easy to read.
Then, if the branches truly do not depend on one another, run them in threads. Wall-clock
time drops because the calls wait on the network together. The number of calls, the tokens
and the bill do not drop at all - and parallel calls hit provider rate limits sooner.

The fan-in step must be bounded: validate fields, merge duplicates, rank, cap the output,
stop. A supervisor that can keep asking for revisions has become another autonomous loop
rather than a controlled aggregation step - and it must report what it merged and what it
truncated, or the cap silently deletes findings.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — The vocabulary, on this exact example

- **Shared state** — what every branch reads. Here it is `SOURCE`, one immutable string. Nothing writes to it, which is precisely why the branches are safe to run at the same time.
- **Fan-out** — one step launches several independent branches. Ours is bounded at three: three roles, three calls, no branch may spawn another.
- **Handoff** — what crosses the boundary between agents. Only `Finding` records: no personas, no chat history, no hidden reasoning.
- **Fan-in** — one step collects every branch's output and combines it. That is the supervisor, and it is the whole of the next notebook.


In [ ]:
from review_team import SPECIALIST_ROLES

print("Bounded fan-out width:", len(SPECIALIST_ROLES))
print("Roles                :", list(SPECIALIST_ROLES))
print()
print()
print("Shared state: one immutable string that every branch reads.")
print("   artifact length (characters):", len(SOURCE))
print("   any branch writing to it? no - each returns its own list of findings")


## Step 3 — What a narrow role actually changes

A specialist is not the same prompt with a new name. Each one answers a different question about the *same* text, and is only allowed to report findings in its own category.


In [ ]:
from review_team import specialist_review

for role in SPECIALIST_ROLES:
    findings = specialist_review(SOURCE, role)
    print(f"\n{role} specialist -> {len(findings)} finding(s)")
    for finding in findings:
        print(f"   line {finding.line:>3} | {finding.severity:<8} | {finding.title}")
    # The role contract: a specialist may not stray outside its category.
    assert all(f.category == role for f in findings)
print("\nEvery finding stayed inside its reviewer's category.")


## Step 4 — Look at one handoff record

This is the entire message the security specialist sends onward. Compare it with "here is my conversation, please read it": it is small, validated, and mergeable.


In [ ]:
handoff = specialist_review(SOURCE, "security")[0]

for field_name, value in handoff.as_dict().items():
    print(f"{field_name:>15}: {value}")
print("\nWhat is NOT in the handoff: the prompt, the reasoning, the chat history.")


## Step 5 — Sequential first, because it is easy to debug

We give the mock reviewer a 0.3 second delay so it behaves like a real network call. Sequential means: call one, wait, call the next, wait, call the last.


In [ ]:
from time import perf_counter
from review_team import MockStructuredReviewer, run_specialist_team

# A reviewer that pretends each call takes 0.3 s of network time.
slow_provider = MockStructuredReviewer("blind_spots", latency_s=0.3)

start = perf_counter()
sequential = run_specialist_team(SOURCE, slow_provider, parallel=False)
sequential_seconds = perf_counter() - start

print("Execution   : sequential")
print("Model calls : %d" % sequential.model_calls)
print("Tokens      : %d" % sequential.total_tokens)
print("Wall clock  : %.2f s" % sequential_seconds)


## Step 6 — Now fan out into threads

`ThreadPoolExecutor` starts the three calls together. While one branch is waiting on the network, the others are waiting too — so the total wait is roughly the *longest* call, not the sum of all three.


In [ ]:
start = perf_counter()
parallel = run_specialist_team(SOURCE, slow_provider, parallel=True)
parallel_seconds = perf_counter() - start

print("Execution   : parallel (ThreadPoolExecutor, max_workers=3)")
print("Model calls : %d" % parallel.model_calls)
print("Tokens      : %d" % parallel.total_tokens)
print("Wall clock  : %.2f s" % parallel_seconds)


## Step 7 — What parallelism did and did not buy

Read the table carefully. One column improves. The two columns you pay money for do not move at all.


In [ ]:
print(f"{'measure':<22}{'sequential':>12}{'parallel':>12}")
print("-" * 46)
print(f"{'wall clock (s)':<22}{sequential_seconds:>12.2f}{parallel_seconds:>12.2f}")
print(f"{'model calls':<22}{sequential.model_calls:>12}{parallel.model_calls:>12}")
print(f"{'tokens':<22}{sequential.total_tokens:>12}{parallel.total_tokens:>12}")
print(f"{'findings reported':<22}{len(sequential.findings):>12}{len(parallel.findings):>12}")
print()
print("Speed-up: %.1fx" % (sequential_seconds / parallel_seconds))
print("Same findings in the same order:",
      [f.id for f in sequential.findings] == [f.id for f in parallel.findings])
print()
print("Parallel execution buys wall-clock time. It does not reduce calls, tokens or cost,")
print("and it reaches a provider's rate limit three times faster.")


### Try it yourself

The pool is capped at 3 workers. Predict the wall clock if we capped it at **1** worker instead, keeping everything else identical.


In [ ]:
# --- Worked solution ---
# Fan out by hand so the bound is visible: max_workers is the whole cap.
from concurrent.futures import ThreadPoolExecutor

def timed_fan_out(max_workers):
    start = perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        # Each branch is one call to one specialist; all read the same shared SOURCE.
        results = list(pool.map(lambda role: slow_provider.review(SOURCE, role),
                                SPECIALIST_ROLES))
    seconds = perf_counter() - start
    total_findings = sum(len(findings) for findings, _usage in results)
    return seconds, total_findings

for workers in (1, 2, 3):
    seconds, total = timed_fan_out(workers)
    print("max_workers=%d -> %.2f s, %d findings, 3 model calls" % (workers, seconds, total))

print()
print("max_workers=1 is sequential execution wearing a thread pool.")
print("The findings and the bill never change; only how long we wait does.")


### Checkpoint

**1. Why is it safe to run these three reviewers at the same time?**

<details><summary>Show answer</summary>

Because the shared state is read-only. Every branch reads the same immutable `SOURCE` string and writes only to its own list of findings, so there is no order in which they could interfere. The moment a branch needed to *modify* shared state, we would need locking or a merge rule, and the simple fan-out would stop being safe.

</details>

**2. Your manager says "run the specialists in parallel to cut our API bill". What do you reply?**

<details><summary>Show answer</summary>

Parallelism does not touch the bill. We measured it: sequential and threaded runs made the same 3 calls and used the same tokens, and only the wall clock fell (about 0.9 s to 0.3 s). To cut cost you must remove calls — for example by proving findings with the AST checker or by using one reviewer instead of three.

</details>

### Recap

- Limitation we saw: running three specialists one after another takes the sum of all three waits.
- Layer we added: a bounded fan-out over read-only shared state, with `Finding` records as the only handoff.
- Evidence it worked: 0.9 s -> 0.3 s wall clock, identical findings, identical 3 calls and identical tokens.


---

### Section 4.4 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-5"></a>

## 4.5 — Supervisor Synthesis


## Before you begin

### Learning outcomes

- Merge findings from four branches with a rule that does not rely on shared ids.
- Make the supervisor report what it merged and what it truncated.
- Break the merge rule on purpose and watch a real defect disappear.

Architecture reference: [Day 4 diagrams D14](../diagrams/source/day_04.md)

### Expected observation

12 findings arrive, id-based merging leaves 3 duplicates, location-based merging removes them — and a wider merge window silently deletes a genuine defect.


## Concept briefing

## Deduplication is harder than matching IDs

If every reviewer reuses the same seeded id, deduplication is a dictionary lookup. Real
reviewers do not. A static checker calls it `AST-EVAL-20`; a model calls it
`MODEL-SEC-20-3`; a human calls it "unsafe eval". Same defect, three ids, three survivors
in your final report.

So the supervisor merges on a rule that does not depend on shared ids: same category, and
lines close enough to be the same place. That rule is *live*, and it can be wrong. Two
genuinely different security defects on adjacent lines will be **falsely merged** and the
second one is lost. Loosen the rule and you hide real defects; tighten it and duplicates
survive. There is no setting that is right for every artifact, which is why production
systems keep a human in the merge loop.

Evaluation needs the same discipline. Credit each known defect **once**: three reviewers
reporting the same problem is one defect found plus two duplicates, never three finds.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — Collect everything the branches produced

Four groups arrive at the fan-in point: the AST checker plus three specialists. Nobody has merged anything yet.


In [ ]:
from review_team import SPECIALIST_ROLES, deterministic_checks, specialist_review

groups = [deterministic_checks(SOURCE)]
groups += [specialist_review(SOURCE, role) for role in SPECIALIST_ROLES]

labels = ["ast_checker", *SPECIALIST_ROLES]
for label, group in zip(labels, groups):
    print(f"{label:<16} produced {len(group)} finding(s)")

print("\nTotal findings arriving at the supervisor:", sum(len(g) for g in groups))


## Step 3 — The naive rule: merge on identifier

The obvious rule is "two findings are the same if their ids match". Watch what it does when the parser and a specialist describe the same defect under different ids.


In [ ]:
from review_team import synthesize_with_report

by_id = synthesize_with_report(groups, merge_by="id")

print("Received          :", by_id.received)
print("Merged duplicates :", by_id.merged_duplicates)
print("Kept              :", by_id.kept)
print("Dropped over cap  :", by_id.dropped_over_cap)
print()

# Find the pairs that survived: same category, same line, different id.
for i, first in enumerate(by_id.findings):
    for second in by_id.findings[i + 1:]:
        if first.category == second.category and first.line == second.line:
            print(f"SURVIVING DUPLICATE on line {first.line}:")
            print(f"   {first.id:<26} ({first.reviewer})")
            print(f"   {second.id:<26} ({second.reviewer})")


## Step 4 — A rule that does not need shared ids

Real reviewers invent their own ids, so the supervisor merges on **category plus line proximity** instead: same category, lines within one of each other, same defect. Now the overlaps collapse.


In [ ]:
by_location = synthesize_with_report(groups, merge_by="location")

print(f"{'rule':<12}{'received':>10}{'merged':>9}{'kept':>7}")
print("-" * 38)
for report in (by_id, by_location):
    print(f"{report.merge_key:<12}{report.received:>10}{report.merged_duplicates:>9}"
          f"{report.kept:>7}")

print("\nFinal report, ranked by severity then line:")
for finding in by_location.findings:
    print(f"   {finding.severity:<8} line {finding.line:>3}  {finding.title}  "
          f"[{finding.reviewer}]")


## Step 5 — Duplicates that survive are a measurable cost

The evaluator credits each defect once, so surviving duplicates show up in their own column. This is what a bad merge rule costs a reader: the same problem, twice, in a report they have to triage.


In [ ]:
from review_team import ReviewRun, evaluate

for report in (by_id, by_location):
    row = evaluate(ReviewRun(f"merge_by_{report.merge_key}", report.findings), GOLDEN_PATH)
    print(f"merge_by={report.merge_key:<10} report length {row['reported_findings']:>2} | "
          f"defects found {row['found']}/9 | duplicates in report {row['duplicates']}")


## Step 6 — Capping the output must not be silent

A supervisor has to terminate, so it caps the report. If it truncates without saying so, findings vanish and nobody knows. Ours records the number dropped.


In [ ]:
capped = synthesize_with_report(groups, max_findings=4, merge_by="location")

print("Kept             :", capped.kept)
print("Dropped over cap :", capped.dropped_over_cap)
print()
print("Kept (highest severity first):")
for finding in capped.findings:
    print(f"   {finding.severity:<8} line {finding.line:>3}  {finding.title}")

dropped_ids = ({f.id for f in by_location.findings} - {f.id for f in capped.findings})
print("\nDropped, and reported as dropped:", sorted(dropped_ids))


## Step 7 — The merge rule can be wrong

Location-based merging has no way to know whether two nearby findings are one defect or two. Here are two genuinely different security problems one line apart. The rule merges them, and the second one is gone.


In [ ]:
from review_team import Finding, synthesize

first  = Finding("A", "security", 15, "SQL built by string concatenation",
                 "query = ... + customer_name", "critical", "Parameterise it.", "sec_a")
second = Finding("B", "security", 16, "Query result returned without authorisation check",
                 "return database.execute(query)", "high", "Check the caller.", "sec_b")

merged = synthesize([[first, second]], merge_by="location")

print("Two different defects went in:")
for f in (first, second):
    print(f"   line {f.line}  {f.title}")
print("\nCame out of the supervisor:", len(merged), "finding(s)")
for f in merged:
    print(f"   line {f.line}  {f.title}")
print("\nFALSE MERGE: a real defect was deleted by the deduplication rule.")


### Try it yourself

Widen the merge window from 1 line to 5 and predict what happens to the three correctness defects on lines 7, 9 and 25.


In [ ]:
# --- Worked solution ---
correctness = specialist_review(SOURCE, "correctness")
print("Correctness findings before merging:")
for f in correctness:
    print(f"   line {f.line:>3}  {f.id}  {f.title}")

for window in (1, 5):
    report = synthesize_with_report([correctness], merge_by="location", line_window=window)
    row = evaluate(ReviewRun("demo", report.findings), GOLDEN_PATH)
    print(f"\nline_window={window}: kept {report.kept}, merged {report.merged_duplicates}, "
          f"defects credited {row['found']}")
    for f in report.findings:
        print(f"   line {f.line:>3}  {f.title}")

print()
print("With window=5, lines 7 and 9 are treated as one defect and recall falls.")
print("Loosen the rule and you hide real defects; tighten it and duplicates survive.")
print("There is no window that is right for every artifact - which is why production")
print("systems keep a human in the merge loop.")


### Checkpoint

**1. Why can't the supervisor just deduplicate on the finding id?**

<details><summary>Show answer</summary>

Because ids are only shared inside this classroom. A static checker calls the line-20 problem `AST-EVAL-20`, a model calls it `MODEL-SEC-20-3`, a human calls it "unsafe eval". We measured it: id-based merging left 3 duplicate pairs in the report that location-based merging removed.

</details>

**2. The supervisor caps the report at N findings. What is the minimum it owes the reader when it hits that cap?**

<details><summary>Show answer</summary>

The number it dropped. A cap is a legitimate way to terminate, but silent truncation turns "we found nothing else" and "we stopped looking" into the same output. Our `SynthesisReport` records `dropped_over_cap`, and the run puts it in the trace.

</details>

### Recap

- Limitation we saw: four branches produced 12 findings for 9 defects, and id-based merging left 3 duplicates in the report.
- Layer we added: a bounded fan-in that merges on category plus line proximity, ranks, caps, and reports both `merged_duplicates` and `dropped_over_cap`.
- Evidence it worked: 12 -> 9 findings with 0 duplicates — and a deliberate false merge showing exactly how the same rule can delete a real defect.


---

### Section 4.5 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-6"></a>

## 4.6 — Comparative Evaluation


## Before you begin

### Learning outcomes

- Score all three architectures on the same artifact with the same reviewer.
- Run the comparison twice, against two different reviewers, and read who won.
- State the conditions under which a multi-agent team is not worth building.

Architecture reference: [Day 4 diagrams D15](../diagrams/source/day_04.md)

### Expected observation

Two tables. In the first the specialist team wins 9/9 against 5/9. In the second every system finds 8/9 and the team simply costs three times as much.


## Concept briefing

## Evaluating nondeterministic systems

Do not assert exact model wording. Assert invariants and measure outcomes:

- Is every finding structurally valid?
- Does the evidence quote the supplied artifact?
- How many known defects were found, counted once each?
- How many unsupported findings (false positives) were reported?
- How many duplicates survived synthesis, and how many did the supervisor merge?
- How many calls and tokens were used, according to the provider?
- Did the system terminate within its bounds, and what did it truncate?

Telemetry must be measured, never invented. A step that makes no model call reports zero
tokens and says "no model call"; it does not borrow a plausible-looking estimate. A tidy
number in a results table that nothing actually produced is worse than no number.

One run is an anecdote. Repeat model experiments with the same model, prompt version,
temperature and artifact, and report the variance rather than the best result.

## Capability can change the architecture conclusion

A weaker instruction-following model can benefit a lot from narrow prompts. A stronger
model may handle the general review well enough that the specialist calls add nothing.
So "multi-agent is better" usually means "decomposition compensated for *this* reviewer on
*this* task with *this* prompt."

Our two scenarios make that explicit and measurable rather than rhetorical. In the
`blind_spots` scenario the team finds 9/9 where one reviewer finds 5/9, and the extra
calls are clearly worth it. In the `strong_generalist` scenario the team finds exactly
what one reviewer already found, for three times the calls and tokens, plus duplicates to
merge and three times the surface area to debug. Same code, same artifact, opposite
verdict.

Multi-agent is not worth it when: one reviewer already reaches the quality bar; the extra
recall costs more than the defects it catches; the sub-tasks are not genuinely
independent; or the failure you keep hitting is a capability gap that a narrower prompt
cannot fill.

## Cost and latency arithmetic

Approximate run cost as:

```text
sum of input tokens across calls
+ sum of output and reasoning tokens
+ retries
```

Send the same 1,000-token artifact to three specialists and you pay for that input three
times, unless caching or a provider feature changes the arithmetic. Parallel execution can
cut elapsed time while leaving total cost identical or higher.

A fair comparison records recall, false positives, duplicates, calls, tokens, latency,
estimated cost and debugging burden. Then choose the **smallest** system that meets the
quality requirement.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — One function that scores all three systems

The reviewer is a parameter, the architectures are fixed. Changing one thing at a time is what makes this a comparison rather than a story.


In [ ]:
from review_team import (FallbackReviewer, MockStructuredReviewer, OpenRouterReviewer,
                         evaluate, run_checks_plus_reviewer, run_single_reviewer,
                         run_specialist_team)

PRICE_PER_MILLION_TOKENS = 0.15    # illustrative catalogue price, so you can redo the sums

def build_provider(scenario):
    """Live reviewer when a key exists (with a mock safety net), otherwise the mock."""
    if LIVE:
        return FallbackReviewer(OpenRouterReviewer(), MockStructuredReviewer(scenario))
    return MockStructuredReviewer(scenario)

def score_all_systems(scenario):
    """Run the three architectures with the same reviewer and return one row each."""
    provider = build_provider(scenario)
    runs = [run_single_reviewer(SOURCE, provider),
            run_checks_plus_reviewer(SOURCE, provider),
            run_specialist_team(SOURCE, provider)]
    return [evaluate(run, GOLDEN_PATH, price_per_million_tokens=PRICE_PER_MILLION_TOKENS)
            for run in runs]

COLUMNS = ["system", "found", "recall", "false_positives", "duplicates",
           "merged_duplicates", "model_calls", "tokens", "estimated_cost_usd"]

def cell_text(row, column):
    """Format one table cell; costs get fixed decimals so the column lines up."""
    if column == "estimated_cost_usd":
        return "%.6f" % row[column]
    return str(row[column])

def print_table(scenario, rows):
    print("Scenario:", scenario)
    widths = [max([len(col)] + [len(cell_text(row, col)) for row in rows]) for col in COLUMNS]
    print(" | ".join(col.ljust(w) for col, w in zip(COLUMNS, widths)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(cell_text(row, col).ljust(w) for col, w in zip(COLUMNS, widths)))

print("Scoring function ready. Provider will be:",
      "live model with mock fallback" if LIVE else "MockStructuredReviewer")


## Step 3 — Scenario A: a reviewer with real blind spots

This is the case people have in mind when they reach for a team of agents.


In [ ]:
rows_a = score_all_systems("blind_spots")
print_table("blind_spots", rows_a)

print("\nWhat each system missed:")
for row in rows_a:
    print(f"   {row['system']:<22} {row['missed'] or 'nothing'}")


### Try it yourself

Now we swap in a reviewer that is already strong on its own — same three architectures, same artifact. Before running the next cell, write down which system you think will win, and what "win" should even mean here.


In [ ]:
# --- Worked solution ---
rows_b = score_all_systems("strong_generalist")
print_table("strong_generalist", rows_b)

print()
best_found = max(row["found"] for row in rows_b)
reaching_it = [row for row in rows_b if row["found"] == best_found]
cheapest = min(reaching_it, key=lambda row: row["model_calls"])

print("Most defects any system found :", best_found, "/ 9")
print("Systems that reached that     :", [row["system"] for row in reaching_it])
print("Smallest system that reached it:", cheapest["system"],
      f"({cheapest['model_calls']} call, {cheapest['tokens']} tokens, "
      f"${cheapest['estimated_cost_usd']:.6f})")
print()
print('"Win" is not "found the most". It is "met the quality bar with the least".')
print("Here the team found nothing the single reviewer had not already found,")
print("and charged three times as much to do it.")


## Step 4 — Put both scenarios side by side

Same code, same artifact, opposite verdict. Read the table and say which system you would deploy in each world.


In [ ]:
print(f"{'scenario':<20}{'system':<24}{'found':>6}{'calls':>7}{'tokens':>8}{'cost $':>10}")
print("-" * 75)
for scenario, rows in (("blind_spots", rows_a), ("strong_generalist", rows_b)):
    for row in rows:
        print(f"{scenario:<20}{row['system']:<24}{row['found']:>6}{row['model_calls']:>7}"
              f"{row['tokens']:>8}{row['estimated_cost_usd']:>10.6f}")
    print()

for scenario, rows in (("blind_spots", rows_a), ("strong_generalist", rows_b)):
    single = rows[0]
    team = rows[2]
    gain = team["found"] - single["found"]
    extra = team["tokens"] - single["tokens"]
    print(f"{scenario:<20} team found {gain:+d} defect(s) for {extra:+d} extra tokens "
          f"and {team['model_calls'] - single['model_calls']:+d} extra calls")


## Step 5 — When a multi-agent team is NOT worth it

We now have measured evidence rather than an opinion. Write these down; they are the deliverable of Day 4.


In [ ]:
reasons = [
    ("One reviewer already meets the bar",
     "strong_generalist: single 8/9 vs team 8/9 -> 0 defects gained for 3x the calls."),
    ("The extra recall costs more than it is worth",
     "Compare cost per extra defect against the cost of the defect escaping."),
    ("The sub-tasks are not truly independent",
     "If a branch needs another branch's output, fan-out becomes a fragile pipeline."),
    ("The failure is a capability gap, not an attention gap",
     "DEF-COR-02 was missed by the generalist AND by the correctness specialist: a "
     "narrower prompt cannot supply knowledge the reviewer never had."),
    ("Debugging burden grows with branches",
     "3 branches + a merge rule = 4 places a wrong report can come from, not 1."),
]

for index, (headline, evidence) in enumerate(reasons, 1):
    print(f"{index}. {headline}")
    print(f"   evidence: {evidence}")


## Step 6 — A reference table that works with no network

`data/captured_comparison.json` holds a saved run so this notebook still shows a table if OpenRouter is down or your key has expired. It was produced offline by the mock reviewer, so it is a record of the classroom result — not evidence about any real language model.


In [ ]:
import json

captured_path = PROJECT_ROOT / "data" / "captured_comparison.json"
captured = json.loads(captured_path.read_text(encoding="utf-8"))

print("Note from the file:", captured["note"])
print("Provider used     :", captured["provider"])

# Which table should your write-up cite? Whichever one actually ran.
if LIVE:
    print("Status            : you ran live, so cite YOUR tables above and use this")
    print("                    file only for the runs where a call failed.")
else:
    print("Status            : no API key, so this file and your tables agree by")
    print("                    construction - both came from the mock reviewer.")
print()

for scenario, rows in captured["scenarios"].items():
    print(scenario)
    for row in rows:
        print(f"   {row['system']:<24} found {row['found']}/9 | "
              f"calls {row['model_calls']} | tokens {row['tokens']}")


## Required live observation

Run one single-reviewer and one bounded specialist comparison with the issued model. Preserve raw structured results; use the captured comparison if the service is unavailable.


### Checkpoint

**1. In the `strong_generalist` scenario the specialist team still found 8 defects — as many as any other system. Why is that not a win?**

<details><summary>Show answer</summary>

Because it found nothing the single reviewer had not already found, while making 3 model calls instead of 1 and using roughly three times the tokens. It also produced duplicates that the supervisor had to merge, and three extra places for a bug to hide. The right question is never "which found the most" but "which is the smallest system that meets the bar".

</details>

**2. Both scenarios use exactly the same architecture code. So what actually changed the conclusion?**

<details><summary>Show answer</summary>

The reviewer's capability. `blind_spots` and `strong_generalist` differ only in which defects the reviewer can see. That is why "multi-agent is better" is never a general claim: it means "decomposition compensated for *this* reviewer on *this* task with *this* prompt", and it must be re-measured when any of those change.

</details>

### Recap

- Limitation we saw: a single comparison table can make a team of agents look unconditionally better than one reviewer.
- Layer we added: the same measurement repeated against a second reviewer, plus a captured offline table so the comparison always runs.
- Evidence it worked: 5/9 -> 9/9 in one scenario and 8/9 -> 8/9 at 3x the cost in the other, from identical architecture code.


---

### Section 4.6 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-7"></a>

## 4.7 — Day 4 Project — Engineering Design Review Team


## Before you begin

### Learning outcomes

- Assemble the whole day: provider contract, deterministic checks, bounded fan-out, supervisor fan-in, and measurement.
- Turn a quality requirement into a deployment decision the code can make.
- Write a decision memo that a reviewer could disagree with using your own numbers.

Architecture reference: [Day 4 diagrams D12–D15](../diagrams/source/day_04.md)

### Expected observation

All three systems terminate with structured findings and inspectable traces, and the recommended system changes when the quality bar changes.


## Concept briefing

## What to carry into Day 5

Days 1-4 repeatedly configure providers, validate tool and model output, enforce limits,
fall back safely and record events. Day 5 pulls those repeated responsibilities into
reusable infrastructure while keeping application-specific instructions, tools and policy
in agent configuration.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — Assemble the system

One provider, three architectures, two scenarios. Nothing new is introduced here — this is the day, wired together.


In [ ]:
from review_team import (FallbackReviewer, MockStructuredReviewer, OpenRouterReviewer,
                         evaluate, run_checks_plus_reviewer, run_single_reviewer,
                         run_specialist_team)

SCENARIOS_TO_RUN = ("blind_spots", "strong_generalist")

def build_provider(scenario):
    if LIVE:
        return FallbackReviewer(OpenRouterReviewer(), MockStructuredReviewer(scenario))
    return MockStructuredReviewer(scenario)

def run_everything(scenario):
    provider = build_provider(scenario)
    runs = [run_single_reviewer(SOURCE, provider),
            run_checks_plus_reviewer(SOURCE, provider),
            run_specialist_team(SOURCE, provider)]
    return runs, [evaluate(run, GOLDEN_PATH) for run in runs]

results = {scenario: run_everything(scenario) for scenario in SCENARIOS_TO_RUN}

print("Provider:", "live model with mock fallback" if LIVE else "MockStructuredReviewer")
for scenario, (runs, rows) in results.items():
    print("\n" + scenario)
    for row in rows:
        print(f"   {row['system']:<24} found {row['found']}/9 | calls {row['model_calls']} | "
              f"tokens {row['tokens']} | merged {row['merged_duplicates']} | "
              f"dropped {row['dropped_over_cap']}")


## Step 3 — Inspect a trace end to end

Every system must be debuggable by reading, not guessing. This is the full trace of the largest one.


In [ ]:
team_run = results["blind_spots"][0][2]

print("System:", team_run.system, "\n")
for step in team_run.trace:
    print(step["step"])
    for key, value in step.items():
        if key != "step":
            print("   ", key, "=", value)

print("\nFinal report:")
for finding in team_run.findings:
    print(f"   {finding.severity:<8} line {finding.line:>3}  {finding.title}  "
          f"[{finding.reviewer}]")


## Step 4 — Check the bounds, not the wording

These assertions are about *structure*: the system stayed inside its limits and every claim carries evidence. Quality is measured, never asserted.


In [ ]:
checks_passed = []

for scenario, (runs, rows) in results.items():
    for run, row in zip(runs, rows):
        assert run.model_calls <= 3, "fan-out must stay bounded"
        assert 0.0 <= row["recall"] <= 1.0
        assert row["false_positives"] == 0, "no finding may point outside the artifact"
        assert all(f.evidence.strip() for f in run.findings), "every finding needs evidence"
        assert len(run.findings) <= 20, "the supervisor must cap its report"
        checks_passed.append(f"{scenario}/{run.system}")

print("Structural checks passed for:")
for name in checks_passed:
    print("   ", name)
print("\nNone of these assert that the multi-agent system wins. That is an observation.")


## Step 5 — Turn a requirement into a decision

A deployment choice needs a stated quality bar. Given one, the rule is mechanical: **the smallest system that clears the bar**.


In [ ]:
def recommend(rows, minimum_recall):
    """Smallest system (fewest model calls) whose recall meets the bar."""
    qualifying = [row for row in rows if row["recall"] >= minimum_recall]
    if not qualifying:
        return None
    return min(qualifying, key=lambda row: (row["model_calls"], row["tokens"]))

BAR = 0.85
print(f"Quality bar: recall >= {BAR}\n")
for scenario, (_runs, rows) in results.items():
    choice = recommend(rows, BAR)
    if choice is None:
        print(f"{scenario:<20} no system meets the bar; do not deploy")
    else:
        print(f"{scenario:<20} deploy {choice['system']:<24} "
              f"(recall {choice['recall']}, {choice['model_calls']} call(s), "
              f"{choice['tokens']} tokens)")


### Try it yourself

Raise the quality bar to 1.0 — nothing may be missed. Predict which system gets recommended in each scenario, and whether any recommendation becomes "do not deploy".


In [ ]:
# --- Worked solution ---
for bar in (0.85, 1.0):
    print(f"Quality bar: recall >= {bar}")
    for scenario, (_runs, rows) in results.items():
        choice = recommend(rows, bar)
        verdict = "DO NOT DEPLOY (no system meets the bar)" if choice is None else (
            f"{choice['system']} ({choice['model_calls']} call(s), {choice['tokens']} tokens)")
        print(f"   {scenario:<20} -> {verdict}")
    print()

print("Raising the bar changes the answer, and in strong_generalist it removes every")
print("option: no amount of orchestration finds DEF-COR-02, because neither the general")
print("reviewer nor the correctness specialist can see it. At that point the fix is a")
print("better reviewer, a deterministic check, or a human - not another agent.")


## Step 6 — Your decision memo

One paragraph, using your own numbers from the tables above. The template below prints a filled-in example so you know exactly what is expected.


In [ ]:
rows_blind = results["blind_spots"][1]
single_row, _augmented_row, team_row = rows_blind

memo = f"""DECISION MEMO - Engineering Design Review Team

Chosen system : {team_row['system']} (scenario: blind_spots)
Evidence      : recall {team_row['recall']} ({team_row['found']}/9) versus
                {single_row['recall']} ({single_row['found']}/9) for a single reviewer.
                False positives {team_row['false_positives']}; duplicates surviving
                synthesis {team_row['duplicates']}; the supervisor merged
                {team_row['merged_duplicates']} overlapping findings and dropped
                {team_row['dropped_over_cap']} over its cap.
Cost          : {team_row['model_calls']} model calls and {team_row['tokens']} tokens versus
                {single_row['model_calls']} call and {single_row['tokens']} tokens - about
                {team_row['tokens'] / single_row['tokens']:.1f}x the spend for
                {team_row['found'] - single_row['found']} extra defects.
Latency       : parallel fan-out cut wall clock roughly 3x in notebook 4.4; it did not
                reduce calls or tokens.
Debugging     : 4 branches plus a merge rule instead of 1 call - more places to be wrong.
Reverses if   : the reviewer improves. Measured in the strong_generalist scenario, the
                team found {results['strong_generalist'][1][2]['found']}/9, exactly what one
                reviewer found, for 3x the calls. Then deploy the single reviewer.
"""
print(memo)


### Checkpoint

**1. Your team wants to add a fourth specialist (performance). What must you show before and after adding it?**

<details><summary>Show answer</summary>

The same table, measured both ways: recall, false positives, duplicates, model calls, tokens, latency and the merge report. Adding a branch is justified only if it finds defects the current system misses, at a cost you would still pay knowing the number. The architecture supports it — `SPECIALIST_ROLES` and the supervisor do not change — which is exactly why the discipline has to come from the measurement.

</details>

**2. At a quality bar of 1.0, no system qualifies in the `strong_generalist` scenario. What is the correct engineering response?**

<details><summary>Show answer</summary>

Not "add more agents". The missed defect (DEF-COR-02, a flat discount that can drive a total negative) is invisible to the generalist *and* to the correctness specialist, so it is a capability gap, not an attention gap. The options are a stronger reviewer, a deterministic check or test that encodes the business rule, or a human reviewer for that class of defect.

</details>

### Recap

- Limitation we saw: a system can hit every structural bound and still fail the quality requirement, and more agents will not fix it.
- Layer we added: an explicit quality bar plus a mechanical rule — the smallest system that clears it — and a memo built from measured numbers.
- Evidence it worked: the recommendation flips between scenarios and flips again when the bar moves from 0.85 to 1.0.


---

### Section 4.7 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


<a id="day-4-section-8"></a>

## 4.8 — Pivotal Exercise: Merge Specialist Findings

This is the day's one hands-on implementation lab. It uses no API key. Try the starter cell first; a fully commented reference solution follows the check so you can compare or catch up.


## Why this mechanism matters

Specialists may overlap, disagree, or fail. Deterministic aggregation makes the supervisor boundary inspectable and avoids spending another model call on rules ordinary code can enforce.

## Contract

Ignore results whose `status` is not `ok`, keep the first copy of each finding `id`, and sort the survivors by descending `severity` then ascending `id`.

Before coding, write one sentence predicting the easiest mistake to make.

In [ ]:
def merge_findings(results):
    """Return one deduplicated, ranked list of findings.

    results -> [{"status": "ok", "findings": [{"id": "F1", "severity": 3}, ...]},
                {"status": "error", "error": "timeout"}, ...]
    """
    # TODO: skip results whose status is not "ok"
    # TODO: keep only the first finding seen for each id
    # TODO: sort by (-severity, id)
    raise NotImplementedError("Complete supervisor merge")

## Behavioural check

Run this after completing the starter cell. If you have not finished, it prints a hint instead of failing. A passing check proves the listed contract examples, not every possible input.

In [ ]:
def run_checks():
    results = [
        {"status": "ok", "findings": [{"id": "F2", "severity": 2}, {"id": "F1", "severity": 3}]},
        {"status": "error", "error": "timeout"},
        {"status": "ok", "findings": [{"id": "F1", "severity": 3}, {"id": "F3", "severity": 1}]},
    ]
    merged = merge_findings(results)
    print("Merged:", [(item["id"], item["severity"]) for item in merged])
    assert [item["id"] for item in merged] == ["F1", "F2", "F3"], "dedupe by id, tolerate the failed specialist"

    # Severity must win over id order: Z1 (severity 4) comes before A9 (severity 1).
    tricky = [{"status": "ok", "findings": [{"id": "A9", "severity": 1}, {"id": "Z1", "severity": 4}]}]
    assert [item["id"] for item in merge_findings(tricky)] == ["Z1", "A9"], "sort by severity first, then id"
    print("PASS: merge is deterministic, deduplicated, ranked, and tolerant of partial failure")

try:
    run_checks()
except NotImplementedError:
    print("Not implemented yet. Complete the starter cell above, or study the reference solution below and re-run this cell.")

## Reference solution

Read this even if your check passed: compare each commented line with your version, then re-run the check cell above.

In [ ]:
# --- Reference solution: read it line by line, then re-run the check cell above ---
def merge_findings(results):
    first_seen = {}                                        # id -> finding (first copy wins)
    for result in results:
        if result.get("status") != "ok":                   # a failed specialist must not erase the others
            print("skipping failed specialist:", result.get("error"))
            continue
        for finding in result["findings"]:
            first_seen.setdefault(finding["id"], finding)  # setdefault keeps the first copy
    return sorted(first_seen.values(), key=lambda f: (-f["severity"], f["id"]))  # high severity first, then id

print("Reference merge_findings defined. Re-run the check cell above to see PASS.")

## Explain

**Why can id-based deduplication still miss semantic duplicates?**

<details><summary>Show answer</summary>

Two specialists can describe the same defect with different ids or different wording. A stable key built from category and line number catches more, but it can also falsely merge two different problems on the same line. Day 4.5 shows why the supervisor keeps both evidence references.

</details>

**Why sort by severity before id?**

<details><summary>Show answer</summary>

The report is read top-down by a busy engineer. Ordering by id would bury a critical finding under cosmetic ones; the id is only a tiebreaker to keep the output deterministic.

</details>

---

### Section 4.8 checkpoint

Before continuing, confirm that you can state: **what limitation we observed, what layer we added, and what evidence shows the improvement worked.**


## Day 4 completion checklist

- [ ] I can explain how every section contributes to the **Engineering Design Review Team**.
- [ ] I ran the deterministic path and at least one required live observation or classroom fallback.
- [ ] I attempted the pivotal exercise before reading its reference solution, and I can explain the solution line by line.
- [ ] I can identify the system state, safety boundary, and evidence used to judge the result.
